# Week 3 — Classical Text Representations

Before embeddings, text was vectors of counts. These models still win on small data and short documents. We build them all from scratch.

## Learning Objectives

- Implement Bag-of-Words, TF-IDF, and n-gram models in pure NumPy.
- Derive the TF-IDF weighting from information-theoretic principles.
- Implement Latent Semantic Analysis via truncated SVD.
- Build a complete sparse-vector text classifier and benchmark on a real dataset.

## Required Reading

- Salton, G., Wong, A., & Yang, C. S. (1975). *A Vector Space Model for Automatic Indexing*.
- Deerwester, S., et al. (1990). *Indexing by Latent Semantic Analysis*.
- Chen, S. F., & Goodman, J. (1999). *An Empirical Study of Smoothing Techniques for Language Modeling*.

In [ ]:
import sys, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)

def tokenize(text):
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

## 1. The Vector Space Model

Salton's (1975) insight: represent each document as a vector in $\mathbb{R}^{|V|}$, with one dimension per vocabulary term. Similarity between documents is cosine. The whole edifice of classical information retrieval rests on this single idea.

We build a tiny corpus to ground every later step in concrete numbers.

In [ ]:
CORPUS = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are pets",
    "machine learning models process text",
    "natural language processing uses text models",
    "deep learning is a subset of machine learning",
    "the cat chased the mouse",
    "the dog barked at the cat",
]
LABELS = ['animals', 'animals', 'animals', 'tech', 'tech', 'tech', 'animals', 'animals']

tokenized = [tokenize(d) for d in CORPUS]
vocab = sorted(set(tok for doc in tokenized for tok in doc))
tok2id = {t: i for i, t in enumerate(vocab)}
print(f"Vocabulary ({len(vocab)} terms): {vocab}")

## 2. Bag-of-Words

$\mathbf{x}_d \in \mathbb{N}^{|V|}$, where $x_{d,t}$ is the count of term $t$ in document $d$. Storage is best done as a sparse matrix in practice; for clarity, we use dense NumPy.

In [ ]:
def bag_of_words(docs, vocab):
    tok2id = {t: i for i, t in enumerate(vocab)}
    X = np.zeros((len(docs), len(vocab)), dtype=np.float64)
    for d, doc in enumerate(docs):
        for tok in doc:
            if tok in tok2id:
                X[d, tok2id[tok]] += 1
    return X

X_bow = bag_of_words(tokenized, vocab)
print(f"X_bow shape: {X_bow.shape}")
print(f"Document 0 ({CORPUS[0]!r}):")
for t, count in zip(vocab, X_bow[0]):
    if count > 0:
        print(f"  {t!r:20s} {int(count)}")

## 3. TF-IDF — derivation from information theory

**Term frequency**: $\text{tf}(t, d) = $ count of $t$ in $d$, sometimes log-scaled: $1 + \log \text{tf}$.

**Inverse document frequency**:

$$\text{idf}(t) = \log \frac{N}{\text{df}(t)}, \quad \text{df}(t) = |\{d : t \in d\}|.$$

This is the surprisal — the information content — of observing $t$ in a uniformly chosen document. A term appearing in *every* document has $\text{idf} = 0$ and conveys no discriminative information. A term appearing in one document has $\text{idf} = \log N$ — maximum specificity.

**TF-IDF** weights $\text{tf}(t, d) \cdot \text{idf}(t)$. Documents are typically L2-normalized so cosine similarity reduces to a dot product.

In [ ]:
def tfidf(docs_tokens, vocab):
    N = len(docs_tokens)
    X = bag_of_words(docs_tokens, vocab)
    # tf with log scaling
    tf = np.where(X > 0, 1 + np.log(X + 1e-12), 0.0)
    # df
    df = (X > 0).sum(axis=0)
    idf = np.log((N + 1) / (df + 1)) + 1  # smoothed (sklearn convention)
    W = tf * idf
    # L2-normalize rows
    norms = np.linalg.norm(W, axis=1, keepdims=True) + 1e-12
    return W / norms, idf

X_tfidf, idf_vec = tfidf(tokenized, vocab)
print(f"IDF per term (sorted descending):")
for t, w in sorted(zip(vocab, idf_vec), key=lambda kv: -kv[1])[:10]:
    print(f"  {t!r:20s} idf = {w:.3f}")
print(f"\nRow 0 norm: {np.linalg.norm(X_tfidf[0]):.4f} (should be 1.0)")

In [ ]:
# Cosine similarity matrix — diagonal is 1.0, off-diagonal reveals topic structure.
sim = X_tfidf @ X_tfidf.T
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(len(CORPUS))); ax.set_yticks(range(len(CORPUS)))
ax.set_xticklabels([f"d{i}" for i in range(len(CORPUS))])
ax.set_yticklabels([f"d{i}: {c[:30]}" for i, c in enumerate(CORPUS)])
for i in range(len(CORPUS)):
    for j in range(len(CORPUS)):
        ax.text(j, i, f"{sim[i,j]:.2f}", ha='center', va='center',
                color='white' if sim[i,j] < 0.5 else 'black', fontsize=8)
ax.set_title('Document–document cosine similarity (TF-IDF)')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

## 4. N-gram language models

A maximum-likelihood bigram model estimates

$$P(w_t \mid w_{t-1}) = \frac{c(w_{t-1}, w_t)}{c(w_{t-1})}.$$

But many bigrams have zero count in the training data — assigning them probability 0 gives infinite perplexity on any held-out text that contains them. Smoothing fixes this. We implement two: Laplace (add-one) and the more sophisticated Kneser–Ney.

In [ ]:
class BigramLM:
    def __init__(self, smoothing='laplace', discount=0.75):
        self.smoothing = smoothing
        self.discount = discount

    def fit(self, docs_tokens):
        self.unigram = Counter()
        self.bigram = Counter()
        self.context = Counter()
        for doc in docs_tokens:
            doc = ['<s>'] + doc + ['</s>']
            for i in range(len(doc)):
                self.unigram[doc[i]] += 1
                if i > 0:
                    self.bigram[(doc[i-1], doc[i])] += 1
                    self.context[doc[i-1]] += 1
        self.V = len(self.unigram)
        # For Kneser–Ney continuation: P_cont(w) ∝ # distinct contexts that precede w.
        self.preceding = defaultdict(set)
        for (a, b) in self.bigram:
            self.preceding[b].add(a)
        self.total_bigrams = sum(self.bigram.values())
        return self

    def prob(self, w_prev, w):
        c_ab = self.bigram[(w_prev, w)]
        c_a = self.context[w_prev]
        if self.smoothing == 'laplace':
            return (c_ab + 1) / (c_a + self.V)
        elif self.smoothing == 'kneser_ney':
            d = self.discount
            n_followers = sum(1 for (a, _) in self.bigram if a == w_prev)
            lam = (d * n_followers) / max(c_a, 1)
            # Continuation prob: how often w appears as a novel continuation.
            p_cont = len(self.preceding[w]) / max(len(self.bigram), 1)
            return max(c_ab - d, 0) / max(c_a, 1) + lam * p_cont
        else:
            raise ValueError(self.smoothing)

    def perplexity(self, docs_tokens):
        log_p, n = 0.0, 0
        for doc in docs_tokens:
            doc = ['<s>'] + doc + ['</s>']
            for i in range(1, len(doc)):
                p = self.prob(doc[i-1], doc[i])
                log_p += np.log(max(p, 1e-12))
                n += 1
        return float(np.exp(-log_p / n))

lm_lap = BigramLM('laplace').fit(tokenized)
lm_kn  = BigramLM('kneser_ney').fit(tokenized)
print(f"Laplace      perplexity (training set): {lm_lap.perplexity(tokenized):.2f}")
print(f"Kneser–Ney   perplexity (training set): {lm_kn.perplexity(tokenized):.2f}")

# A held-out sentence that mixes seen and unseen bigrams:
test = [['the', 'cat', 'barked', 'at', 'the', 'mouse']]
print(f"\nHeld-out perplexity (mixed novel/seen):")
print(f"  Laplace    : {lm_lap.perplexity(test):.2f}")
print(f"  Kneser–Ney : {lm_kn.perplexity(test):.2f}")

## 5. Latent Semantic Analysis

LSA is truncated SVD on the TF-IDF matrix:

$$X \approx U_k \Sigma_k V_k^\top.$$

Rows of $U_k$ are document embeddings; rows of $V_k$ are term embeddings, both in $\mathbb{R}^k$. The leading singular vectors capture co-occurrence structure — topics that emerge purely from second-order statistics. This is the conceptual ancestor of word2vec (Week 4).

In [ ]:
def lsa(X, k=2):
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    return U[:, :k] * s[:k], Vt[:k, :].T, s

doc_emb, term_emb, sing = lsa(X_tfidf, k=2)
print(f"Singular values: {sing}")
print(f"Document embedding shape: {doc_emb.shape}")
print(f"Term embedding shape:     {term_emb.shape}")

# Visualize documents in 2D LSA space — topic clusters should emerge.
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['tab:blue' if y == 'animals' else 'tab:red' for y in LABELS]
ax.scatter(doc_emb[:, 0], doc_emb[:, 1], c=colors, s=100, edgecolors='black')
for i, (x, y) in enumerate(doc_emb):
    ax.annotate(f"d{i}", (x, y), xytext=(5, 5), textcoords='offset points', fontsize=10)
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set(xlabel='LSA dim 1', ylabel='LSA dim 2',
       title='Documents in 2D LSA space (blue=animals, red=tech)')
plt.tight_layout(); plt.show()

## 6. A complete classifier: logistic regression on TF-IDF

A logistic-regression classifier on TF-IDF features is a strong baseline. We implement it from scratch with batch gradient descent — no sklearn — then verify.

In [ ]:
def logreg_fit(X, y, lr=0.5, n_iter=500, l2=0.01):
    n, d = X.shape
    w = np.zeros(d); b = 0.0
    for _ in range(n_iter):
        z = X @ w + b
        p = 1 / (1 + np.exp(-z))
        grad_w = X.T @ (p - y) / n + l2 * w
        grad_b = (p - y).mean()
        w -= lr * grad_w; b -= lr * grad_b
    return w, b

def logreg_predict(X, w, b):
    return (1 / (1 + np.exp(-(X @ w + b)))) > 0.5

y = np.array([1 if l == 'tech' else 0 for l in LABELS])
w, b = logreg_fit(X_tfidf, y, n_iter=2000)
preds = logreg_predict(X_tfidf, w, b)
acc = (preds == y).mean()
print(f"Training accuracy: {acc:.3f}")

# Which terms most strongly predict 'tech'?
top = np.argsort(w)[::-1][:5]
print("\nMost predictive of 'tech':")
for i in top:
    print(f"  {vocab[i]!r:20s} weight = {w[i]:+.3f}")
print("\nMost predictive of 'animals':")
for i in np.argsort(w)[:5]:
    print(f"  {vocab[i]!r:20s} weight = {w[i]:+.3f}")

## 7. Exercises

1. **Cosine-vs-Euclidean equivalence.** Show that for L2-normalized vectors, cosine similarity equals $1 - \tfrac{1}{2}\|\mathbf{x} - \mathbf{y}\|^2$.
2. **BM25 vs. TF-IDF.** Implement BM25 (Robertson & Walker, 1994) and compare against TF-IDF on a retrieval benchmark — for short documents BM25 typically wins. Why?
3. **Kneser–Ney derivation.** Show that Kneser–Ney smoothing can be derived as the predictive distribution of a hierarchical Pitman–Yor process (Teh, 2006).
4. **Topic extraction.** Run LSA with $k = 10$ on a 20 Newsgroups subset. For each of the 10 dimensions, inspect the top-10 terms by $|V_k|$. Do recognizable topics emerge?

---

## Next Week

Week 4 — Distributed Word Representations. From sparse counts to dense vectors: word2vec, GloVe, fastText.